# L2d: Build and Test a Unicode Character Table

L2c introduced ASCII, Unicode code points, hexadecimal notation, and UTF-8 strings. In this lab, we use those ideas to build a function that converts a technical label into a table with one row for each character.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Distinguish character count from byte count:__ Compare the characters in a technical label with its UTF-8 bytes, using the `length(...)` and `ncodeunits(...)` functions, and explain why a Unicode string may require more bytes than characters. The Greek letter and the degree symbol each occupy two bytes, so the twenty-character label occupies twenty-two bytes.
> * __Build a table by iterating over a string:__ Implement a documented function that records each character's position, decimal code point, `U+XXXX` code point, and UTF-8 byte count, iterating with the `enumerate(...)` function rather than indexing the string with integer positions. Direct iteration returns complete `Char` values, while an integer position inside a multibyte character is not a valid string index.
> * __Test text-processing behavior:__ Verify the complete interface with regression tests covering ASCII text, Unicode text, the empty string, and a non-string input that must raise an `ArgumentError`. Each category checks a different promise: the one-byte storage pattern, multibyte code-point formatting, the empty-table schema, and input validation.

Let's get started!
___

## Setup, Data, and Prerequisites

The setup file activates the course environment, loads the student implementation from [`src/Compute.jl`](src/Compute.jl), and imports `DataFrames` and `Test`.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [2]:
# Load this lab's file-relative environment, source code, and imports.
include(joinpath(@__DIR__, "Include.jl"));

The setup loads [the `DataFrames.jl` package](https://dataframes.juliadata.org/stable/), which stores the character table, [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), which provides the testing macros, and the `L2dUnicodeTable` module from [`src/Compute.jl`](src/Compute.jl), which exports the `character_table(...)` function you will complete in Task 2.

___

## Task 1: Examine the characters and bytes in a technical label

Technical text often contains special characters, including the Greek capital letters such as $\Delta$ or the degree $\degree$ symbol, etc. Julia stores all text as a `String` composed of UTF-8 code units, and each UTF-8 code unit is one byte. ASCII characters (letters, digits, punctuation, and control characters) require one byte each, while special characters like `Δ` and `°` require more than one byte.

[The `length(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.length) counts characters when applied to a string. [The `ncodeunits(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.ncodeunits) counts its UTF-8 bytes. [The `collect(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.collect-Tuple%7BAny%7D) produces a `Vector{Char}`, while [the `codeunits(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.codeunits) exposes the stored byte values for the string.

Let's examine the label `ΔP = 25 kPa at 80 °C` in Julia:

In [ ]:
technical_label = "ΔP = 25 kPa at 80°C"

# Collect the logical characters separately from the underlying UTF-8 bytes.
label_characters = collect(technical_label)
label_bytes = collect(codeunits(technical_label))

# Compare the two counts and retain both representations for inspection.
(
    character_count = length(technical_label),
    byte_count = ncodeunits(technical_label),
    characters = label_characters,
    bytes = label_bytes,
)

(character_count = 20, byte_count = 22, characters = ['Δ', 'P', ' ', '=', ' ', '2', '5', ' ', 'k', 'P', 'a', ' ', 'a', 't', ' ', '8', '0', ' ', '°', 'C'], bytes = UInt8[0xce, 0x94, 0x50, 0x20, 0x3d, 0x20, 0x32, 0x35, 0x20, 0x6b  …  0x20, 0x61, 0x74, 0x20, 0x38, 0x30, 0x20, 0xc2, 0xb0, 0x43])

The two counts differ because the string contains multibyte characters. This also explains why a Julia `String` should not be processed with `for index in 1:length(text)`. Integer positions inside a multibyte character are not valid string indices.

Iterating directly over the string returns complete `Char` values. Pairing that iteration with [the `enumerate(...)` function](https://docs.julialang.org/en/v1/base/iterators/#Base.Iterators.enumerate) provides the consecutive character positions needed for the output table.

In [ ]:
# Record the first five character positions without indexing into the String.
first_five_characters = collect(Iterators.take(enumerate(technical_label), 5))

___

## Task 2: Implement the character table function

Task 1 inspected the label's characters, bytes, and positions one expression at a time. Now, let's write a function that records these values, together with each character's code point, in a single table makes for any input string.

[The `character_table(...)` function](src/Compute.jl) takes one string and returns a table with one row per character. As in the L2b lab, the contract states what the caller supplies, what the function promises to return, and the error it raises for an input outside the interface.

> __Input__
>
> * `text::AbstractString`: the text to analyze. The empty string is supported and returns an empty table with the documented columns.
>
> __Output__
>
> A `DataFrame` with one row per character and five typed columns:
>
> * `position::Int`: the character's consecutive position in the text, beginning at 1. This is a character count, not a byte index.
> * `character::Char`: the Julia `Char` value at that position.
> * `decimal_codepoint::Int`: the character's Unicode code point as a base-10 integer.
> * `unicode_codepoint::String`: the same code point in uppercase `U+XXXX` notation.
> * `utf8_byte_count::Int`: the number of UTF-8 bytes used to store that character.
>
> __Errors__
>
> * [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError): the input is not a string. Validation raises this error before any calculation begins.

Open [`src/Compute.jl`](src/Compute.jl) and complete its three `TODO` sections to implement the character table function. Then restart the kernel and run the notebook from the beginning so that Julia loads the revised module. Until every section is complete, calling `character_table(...)` with any string raises a direct implementation error.

Let's build the table for the technical label from Task 1:

In [ ]:
# Build the complete table, then leave it as the cell's final expression so
# the notebook displays every row and column.
technical_label_table = character_table(technical_label)
technical_label_table

The first row should describe `Δ` as decimal code point `916`, hexadecimal code point `U+0394`, and a two-byte UTF-8 character. The degree symbol should appear later as `U+00B0`, also with a two-byte UTF-8 representation. ASCII letters, digits, spaces, and punctuation should each use one byte.

In [ ]:
# Select the two non-ASCII rows for a direct comparison.
non_ascii_rows = filter(row -> row.utf8_byte_count > 1, technical_label_table)

___

## Task 3: Test the complete interface

Task 2 confirmed the character table by building it for one label and inspecting a few rows by hand, but nothing repeats that inspection when the implementation changes: a later edit to [`src/Compute.jl`](src/Compute.jl) could replace direct iteration with integer-position indexing, which handles ASCII text correctly and fails only when a multibyte character such as `Δ` arrives. So how do we guard against this type of issue?

Regression tests guard against this by recording the behavior that is correct today as executable checks; rerunning them after any change either confirms that the function is still working correctly or reports exactly what is broken.

Julia's [`Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) provides the macros for writing these checks.

| Test tool | Purpose in this lab |
|:--|:--|
| [`@testset`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@testset) | Groups the character-table checks and prints one summary. |
| [`@test`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) | Verifies table columns, code points, byte counts, and empty-string behavior. |
| [`@test_throws`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test_throws) | Verifies that a non-string input raises the documented exception type. |

The ASCII case verifies the simplest storage pattern. The Unicode case verifies multibyte characters and code-point formatting. The empty-string case verifies that the function returns the documented schema even when there are no rows. The non-string case verifies that validation rejects an input outside the interface before any calculation begins.

In [ ]:
@testset "Unicode character table interface" begin
    # Verify the schema and contents for text containing only ASCII characters.
    ascii_table = character_table("P101")
    @test names(ascii_table) == [
        "position",
        "character",
        "decimal_codepoint",
        "unicode_codepoint",
        "utf8_byte_count",
    ]
    @test ascii_table.position == [1, 2, 3, 4]
    @test ascii_table.character == ['P', '1', '0', '1']
    @test ascii_table.utf8_byte_count == [1, 1, 1, 1]

    # Verify code points and byte counts for Greek and technical symbols.
    unicode_table = character_table("ΔP °C")
    @test unicode_table.character == ['Δ', 'P', ' ', '°', 'C']
    @test unicode_table.decimal_codepoint[[1, 4]] == [916, 176]
    @test unicode_table.unicode_codepoint[[1, 4]] == ["U+0394", "U+00B0"]
    @test unicode_table.utf8_byte_count == [2, 1, 1, 2, 1]

    # Verify the row count and typed schema for an empty string.
    empty_table = character_table("")
    @test nrow(empty_table) == 0
    @test eltype(empty_table.character) == Char
    @test eltype(empty_table.unicode_codepoint) == String

    # Verify that an input outside the documented interface is rejected.
    @test_throws ArgumentError character_table(42)
end

___

## Summary

In this lab, we examined the UTF-8 representation of a technical label, implemented a function that converts text into a Unicode character table, and tested the complete interface.

> __Key Takeaways:__
>
> * **Characters and bytes are different units:** A Unicode string can contain fewer characters than bytes because UTF-8 stores many non-ASCII code points in multiple bytes; the technical label holds twenty characters in twenty-two bytes. Code that treats the two counts as interchangeable carries a latent failure that appears only when non-ASCII text arrives.
> * **Direct iteration preserves character boundaries:** Iterating over a string returns complete `Char` values in order, and pairing that iteration with the `enumerate(...)` function supplies the consecutive character positions the table records. Looping over integer positions up to the character count can instead place an index inside a multibyte character, which fails only when the input happens to contain one.
> * **An empty result still needs a defined schema:** Allocating the five typed columns before processing the input gives empty and populated results the same documented structure, so a caller can rely on the schema without checking the row count first. The tests confirm the empty table keeps typed columns even when there are no rows.

These practices apply when technical labels, units, symbols, or data files contain text that extends beyond basic ASCII.

___